In [ ]:
import pandas as pd

train = pd.read_csv("/home/asusesra/sms-fraud-detection/data/processed/tr_train.csv", keep_default_na=False)
print(train.shape)
train.head()

In [ ]:
print(train["ikili"].value_counts())
print()
print(train["fold"].value_counts().sort_index())

In [ ]:
print("Belirsiz mesaj sayisi:", (train["belirsiz"] == "e").sum())

train = train[train["belirsiz"] != "e"].reset_index(drop=True)
print(train.shape)

In [ ]:
X = train["mesaj"]
y = (train["ikili"] == "spam").astype(int)

print(X.shape, y.shape)
print(y.value_counts())

In [ ]:
import sys
sys.path.append("/home/asusesra/sms-fraud-detection")

from src.temizle import temizle

for m in X.head(5):
    print("ÖNCE :", m[:90])
    print("SONRA:", temizle(m)[:90])
    print()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_temiz = [temizle(m) for m in X]

vek = TfidfVectorizer(analyzer="word", lowercase=False)
M = vek.fit_transform(X_temiz)

print("Matris boyutu:", M.shape)

In [ ]:
kelimeler = vek.get_feature_names_out()
satir = M[0]

print("Mesaj:", X_temiz[0])
print()

sirali = sorted(zip(kelimeler[satir.nonzero()[1]], satir.data.round(3)), key=lambda t: -t[1])
for kelime, puan in sirali:
    print(f"{kelime:<15} {puan}")

In [ ]:
vek_harf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), lowercase=False, min_df=2)
M2 = vek_harf.fit_transform(X_temiz)
print("Matris boyutu:", M2.shape)

harfler = vek_harf.get_feature_names_out()
satir = M2[0]
sirali = sorted(zip(harfler[satir.nonzero()[1]], satir.data.round(3)), key=lambda t: -t[1])
for parca, puan in sirali[:10]:
    print(f"{parca!r:<12} {puan}")

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

egit = train["fold"] != 0
dene = train["fold"] == 0

X_egit = [temizle(m) for m in X[egit]]
X_dene = [temizle(m) for m in X[dene]]

vek = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), lowercase=False, min_df=2)
A = vek.fit_transform(X_egit)
B = vek.transform(X_dene)

model = LinearSVC(class_weight="balanced", random_state=55)
model.fit(A, y[egit])
tahmin = model.predict(B)

print(classification_report(y[dene], tahmin, target_names=["meşru", "spam"], digits=3))

In [ ]:
from sklearn.metrics import f1_score

puanlar = []
for kat_no in sorted(train["fold"].unique()):
    egit = train["fold"] != kat_no
    dene = train["fold"] == kat_no

    X_egit = [temizle(m) for m in X[egit]]
    X_dene = [temizle(m) for m in X[dene]]

    vek = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), lowercase=False, min_df=2)
    A = vek.fit_transform(X_egit)
    B = vek.transform(X_dene)

    model = LinearSVC(class_weight="balanced", random_state=55)
    model.fit(A, y[egit])
    f1 = f1_score(y[dene], model.predict(B))
    puanlar.append(f1)
    print(f"Kat {kat_no}: spam F1 = {f1:.3f}")

print(f"\nOrtalama: {sum(puanlar) / len(puanlar):.3f}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import PredefinedSplit, cross_val_score
from src.temizle import Temizleyici

p = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), lowercase=False, min_df=2)),
    ("model", LinearSVC(class_weight="balanced", random_state=55)),
])

kat = PredefinedSplit(train["fold"])
skor = cross_val_score(p, X, y, cv=kat, scoring="f1")
print(skor.round(3), "ortalama:", skor.mean().round(3))

In [ ]:
from sklearn.model_selection import GridSearchCV

ayarlar = {
    "temizle__ascii": [False, True],
    "vek__ngram_range": [(2, 5), (3, 5)],
    "model__C": [0.1, 1, 10],
}

gs = GridSearchCV(p, ayarlar, cv=kat, scoring="f1", n_jobs=-1)
gs.fit(X, y)

print("En iyi ayar:", gs.best_params_)
print("En iyi CV spam F1:", round(gs.best_score_, 3))

tablo = pd.DataFrame(gs.cv_results_)[["params", "mean_test_score", "std_test_score"]]
print(tablo.sort_values("mean_test_score", ascending=False).round(3).to_string())

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix

tahmin = cross_val_predict(gs.best_estimator_, X, y, cv=kat)
print("satır = gerçek, sütun = tahmin (0 meşru, 1 spam)")
print(confusion_matrix(y, tahmin))

kacirilan = train[((y == 1) & (tahmin == 0)).values]     # spam'i meşru sandı
yanlis_alarm = train[((y == 0) & (tahmin == 1)).values]  # meşru'yu spam sandı
print("\nKAÇIRILAN SPAM:", len(kacirilan))
for _, r in kacirilan.head(10).iterrows():
    print(r["id"], r["etiket"], "|", r["mesaj"][:80])
print("\nYANLIŞ ALARM:", len(yanlis_alarm))
for _, r in yanlis_alarm.head(10).iterrows():
    print(r["id"], r["etiket"], "|", r["mesaj"][:80])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve
from sklearn.svm import LinearSVC

p = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), lowercase=False, min_df=2)),
    ("model", LinearSVC(class_weight="balanced", random_state=55)),
])

boyutlar, egit_skor, dene_skor = learning_curve(
    p, X, y, cv=kat, scoring="f1",
    train_sizes=np.linspace(0.1, 1.0, 8), shuffle=True, random_state=55, n_jobs=-1)

plt.plot(boyutlar, egit_skor.mean(axis=1), "o-", label="eğitim skoru")
plt.plot(boyutlar, dene_skor.mean(axis=1), "o-", label="sınav skoru (CV)")
plt.xlabel("eğitimde kullanılan mesaj sayısı")
plt.ylabel("spam F1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.savefig("ogrenme_egrisi.png", dpi=150, bbox_inches="tight")